In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


X = pd.read_csv("../data/secom.data", sep=" ", header=None)
y = pd.read_csv("../data/secom_labels.data", sep =" ", header=None,
                names =["label", "timestamp"])
y["timestamp"] = pd.to_datetime(y["timestamp"], format="%d/%m/%Y %H:%M:%S")

# the label 0/1 ( 1 = fail) and sort by time
df = X.copy()
df["label"]= (y["label"] == 1).astype(int)
df["timestamp"] = y["timestamp"]
df = df.sort_values("timestamp").reset_index(drop=True)

# time  based split
# 80% for training 
# 20% for testing
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

feature_cols = [c for c in df.columns if c not in ("label", "timestamp")]
X_train, y_train = train[feature_cols], train["label"]
X_test, y_test = test[feature_cols], test["label"]

# drop useless columns
keep = [c for c in feature_cols
        if X_train[c].isna().mean() <= 0.5 and X_train[c].nunique() > 1]
X_train, X_test = X_train[keep], X_test[keep]
print(f"features: {len(feature_cols)} -> {len(keep)}")



features: 590 -> 444


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_validate

cv = TimeSeriesSplit(n_splits=5)

def make(model, k= None):
    steps = [( "imp", SimpleImputer(strategy="median")),
             ("sc", StandardScaler())]

    if k:
        steps.append(("sel", SelectKBest(f_classif, k =k)))
    steps.append(("clf", model))
    return Pipeline(steps)

candidates = {
    "logreg, all features": make(LogisticRegression(class_weight="balanced", C=0.01, max_iter=2000)),
    "logreg, top 20":       make(LogisticRegression(class_weight="balanced", C=0.1, max_iter=2000), k=20),
    "logreg, top 50":       make(LogisticRegression(class_weight="balanced", C=0.1, max_iter=2000), k=50),
    "random forest":        make(RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                        class_weight="balanced_subsample",
                                                        random_state=42, n_jobs=-1)),
}

for name, pipe in candidates.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring=["roc_auc", "average_precision"])
    print(f"{name:22s} ROC-AUC={res['test_roc_auc'].mean():.3f} (±{res['test_roc_auc'].std():.3f})  "
          f"PR-AUC={res['test_average_precision'].mean():.3f} (±{res['test_average_precision'].std():.3f})")

logreg, all features   ROC-AUC=0.514 (±0.084)  PR-AUC=0.075 (±0.040)
logreg, top 20         ROC-AUC=0.570 (±0.020)  PR-AUC=0.110 (±0.060)
logreg, top 50         ROC-AUC=0.569 (±0.075)  PR-AUC=0.118 (±0.049)
random forest          ROC-AUC=0.597 (±0.130)  PR-AUC=0.119 (±0.058)
